In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [14]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride = 1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels, # Number of channels gonna input has (RGB = 3 channels) [depth of the input]
            out_channels, # Number of filters 
            kernel_size = 3,
            stride = stride,
            padding = 1,
            bias = False # This false because we use BatchNorm2d.(It has learnable shift) Therefore use bias unneceessary.
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size = 3,
            stride = 1,
            padding = 1,
            bias = False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)


        if stride != 1 or in_channels != out_channels: 
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size = 1,
                    stride = stride,
                    bias = False
                ),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()


    def forward(self, x):
        identity = x

        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out += self.shortcut(identity)
        out = F.relu(out)

        return out

In [20]:
class ResNet18(nn.Module):
    def __init__(self, in_channels = 3):
        super().__init__()
        self.conv0 = nn.Conv2d(
            in_channels,
            out_channels = 64,
            kernel_size = 7,
            stride = 2,
            padding = 3,
            bias = False
        )
        self.bn0 = nn.BatchNorm2d(64)
        self.maxpool0 = nn.MaxPool2d(
            kernel_size = 3,
            stride = 2,
            padding = 1
        )

        self.resBlock0 = nn.Sequential(
            ResidualBlock(64, 64),
            ResidualBlock(64, 64)
        )
        self.resBlock1 = nn.Sequential(
            ResidualBlock(64, 128, stride = 2),
            ResidualBlock(128, 128)
        )
        self.resBlock2 = nn.Sequential(
            ResidualBlock(128, 256, stride = 2),
            ResidualBlock(256, 256)
        )
        self.resBlock3 = nn.Sequential(
            ResidualBlock(256, 512, stride = 2),
            ResidualBlock(512, 512)
        )

        self.avgpool0 = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(512, 1000) # Input fetures 512 map to the 1000 classes

    def forward(self, x):
        x = F.relu(self.bn0(self.conv0(x)))
        x = self.maxpool0(x)

        x = self.resBlock0(x)
        x = self.resBlock1(x)
        x = self.resBlock2(x)
        x = self.resBlock3(x)

        x = self.avgpool0(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x